# LLM Evaluation — Comparative Opinion Mining

Evaluate popular LLMs on:
- **t5-camera-coqe-data** (English)
- **vcom-data** (Vietnamese)

Notebook runs on both **Google Colab** and **Kaggle**.

### Before running:
1. Add your **OPENROUTER_API_KEY** to secrets:
   - Colab: `Secrets` (key icon in left panel)
   - Kaggle: `Settings → Secrets`
2. Upload `project.zip` (the full project folder as a ZIP) when prompted in Cell 2,  
   **or** set `GITHUB_REPO` to your repo URL to clone automatically.

Get an OpenRouter key at https://openrouter.ai — free tier is enough for benchmarking.

In [ ]:
# ── Cell 1 · Environment & path detection ──────────────────────────────
import sys, os

IN_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IN_KAGGLE = os.path.exists('/kaggle/working')

ENV = 'Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local')
print(f'Running on: {ENV}')

if IN_COLAB:
    WORK_DIR = '/content/msc-project'
elif IN_KAGGLE:
    WORK_DIR = '/kaggle/working/msc-project'
else:
    WORK_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

LLMEVAL_DIR = os.path.join(WORK_DIR, 'llm_eval')
print(f'Project root : {WORK_DIR}')
print(f'LLM eval dir : {LLMEVAL_DIR}')

In [ ]:
# ── Cell 2 · Upload / clone project ────────────────────────────────────
#
# Option A (GitHub) — set GITHUB_REPO and run this cell:
GITHUB_REPO = ''  # e.g. 'https://github.com/haiyan/msc-project.git'
#
# Option B (ZIP upload) — leave GITHUB_REPO empty, then upload project.zip when prompted.

import subprocess, shutil

if not os.path.exists(LLMEVAL_DIR):
    if GITHUB_REPO:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, WORK_DIR], check=True)
        print('Cloned from GitHub.')
    elif IN_COLAB:
        from google.colab import files
        print('Upload your project.zip below:')
        uploaded = files.upload()         # user picks project.zip
        zip_name = list(uploaded.keys())[0]
        import zipfile
        with zipfile.ZipFile(zip_name) as zf:
            zf.extractall('/content/')
        # Handle single top-level folder inside zip
        extracted = [d for d in os.listdir('/content/') if os.path.isdir(f'/content/{d}') and d != 'sample_data']
        if extracted and not os.path.exists(WORK_DIR):
            shutil.move(f'/content/{extracted[0]}', WORK_DIR)
        print(f'Extracted to {WORK_DIR}')
    elif IN_KAGGLE:
        print('On Kaggle: add your project as an Input Dataset (Code > + Add Input) and set WORK_DIR to its path.')
        print('Or upload datasets manually and adjust WORK_DIR above.')
else:
    print(f'Project already at {WORK_DIR}')

In [ ]:
# ── Cell 3 · Install dependencies ──────────────────────────────────────
import subprocess
reqs = os.path.join(LLMEVAL_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', reqs, '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 4 · API key setup ──────────────────────────────────────────────
if IN_COLAB:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
elif IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')

assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found in secrets!'
print('API key loaded.')

In [ ]:
# ── Cell 5 · Configuration ──────────────────────────────────────────────
# Edit models/settings below, then run Cell 6 to start evaluation.

DATASETS = 't5-camera-coqe-data,vcom-data'   # both, or pick one
SPLIT    = 'test'                             # train | dev | test

MODELS = [
    'openai/gpt-4o-mini',
    'anthropic/claude-3.5-haiku',
    'google/gemini-2.0-flash-001',
    'deepseek/deepseek-chat',
    'qwen/qwen-2.5-72b-instruct',
    'meta-llama/llama-3.3-70b-instruct',
]

TEMPERATURE       = 0.0    # deterministic output
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS     = 0.3    # polite delay between API calls
LIMIT             = 0      # set >0 to test on first N samples only

DATASETS_ROOT = os.path.join(WORK_DIR, 'datasets')
OUTPUT_DIR    = os.path.join(WORK_DIR, 'llm_eval', 'results')
CACHE_DIR     = os.path.join(WORK_DIR, 'llm_eval', 'cache')

print('Config ready. Run Cell 6 to start.')

In [ ]:
# ── Cell 6 · Run evaluation ─────────────────────────────────────────────
import subprocess

cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets',          DATASETS,
    '--split',             SPLIT,
    '--models',            *MODELS,
    '--base-url',          'https://openrouter.ai/api/v1',
    '--api-key-env',       'OPENROUTER_API_KEY',
    '--temperature',       str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds',     str(SLEEP_SECONDS),
    '--datasets-root',     DATASETS_ROOT,
    '--output-dir',        OUTPUT_DIR,
    '--cache-dir',         CACHE_DIR,
]
if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

result = subprocess.run(cmd, text=True, capture_output=False)
print('\nExit code:', result.returncode)

In [ ]:
# ── Cell 7 · Display summary table ─────────────────────────────────────
import json, pathlib

summary_file = pathlib.Path(OUTPUT_DIR) / f'summary__{SPLIT}.json'

if summary_file.exists():
    with open(summary_file) as f:
        rows = json.load(f)
    try:
        import pandas as pd
        records = []
        for r in rows:
            records.append({
                'dataset':         r['dataset'],
                'model':           r['model'],
                'E-T5-MACRO-F1':  round(r.get('E-T5-MACRO-F1', 0), 4),
                'E-T4-F1':        round(r.get('E-T4-F1', 0), 4),
                'E-CEE-MICRO-F1': round(r.get('E-CEE-MICRO-F1', 0), 4),
            })
        df = pd.DataFrame(records).sort_values(
            ['dataset', 'E-T5-MACRO-F1'], ascending=[True, False]
        )
        print(df.to_string(index=False))
    except ImportError:
        hdr = f"{'Dataset':<28} {'Model':<45} {'E-T5-MACRO':>12} {'E-T4':>8} {'E-CEE-MICRO':>12}"
        print(hdr)
        print('-' * len(hdr))
        for r in rows:
            print(f"{r['dataset']:<28} {r['model']:<45} "
                  f"{r.get('E-T5-MACRO-F1', 0):>12.4f} "
                  f"{r.get('E-T4-F1', 0):>8.4f} "
                  f"{r.get('E-CEE-MICRO-F1', 0):>12.4f}")
else:
    print('Summary file not found. Check errors above.')


In [ ]:
# ── Cell 8 · Download results (Colab only) ──────────────────────────────
import zipfile, pathlib

if IN_COLAB:
    from google.colab import files
    results_path = pathlib.Path(OUTPUT_DIR)
    zip_path     = '/content/llm_eval_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in results_path.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(results_path.parent))
    files.download(zip_path)
    print('Download started.')
elif IN_KAGGLE:
    print(f'Results are saved to {OUTPUT_DIR}')
    print('Use the "Data" tab in the notebook output to download.')